In [1]:
import pandas as pd
import ast
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from imblearn.over_sampling import SMOTE
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from transformers import AutoTokenizer
from nltk.tokenize import word_tokenize
import logging
import math
from tqdm import tqdm
from joblib import Parallel, delayed
from nltk.corpus import wordnet
from nltk import word_tokenize, pos_tag

In [2]:
# ISO Latin-1 encoding
df = pd.read_csv("Sentiment_Data.csv", encoding='iso-8859-1')

In [3]:
pd.set_option('display.max_columns', None)

In [4]:
df.sample(10)

,Tweet,Sentiment
179846,You know things are crazy when you get an emai...,Strong_Pos
367618,#FreedomConvoy https://t.co/JhXpcDQTdg,Neutral
4118,@tonka023 @CharlieAngusNDP I was a liberal for...,Mild_Neg
269214,"Ben Lynch, a naturopathic doctor from Seattle,...",Neutral
410175,Shared the convoy info to a local fb group and...,Strong_Pos
9206,ð³ð±ððð¯â¤ï¸\n\nThis morning at ...,Mild_Pos
174512,@WentlandRob @AHavrilla @FrankFigliuzzi1 I'm u...,Neutral
283736,@RiverWardRiley Did they have an anti-freedom ...,Mild_Neg
95022,Please vote. Have your say: Do you support Pri...,Strong_Pos
387084,"Those of you who live in Canada, or if you are...",Strong_Pos


In [5]:
df['Sentiment'].value_counts()

Sentiment
Strong_Pos    233700
Neutral        77016
Mild_Pos       64004
Strong_Neg     42556
Mild_Neg       34056
Name: count, dtype: int64

In [6]:
df.duplicated().sum()

36

In [7]:
df = df.drop_duplicates()

In [8]:
df.duplicated().sum()

0

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 451296 entries, 0 to 451331
Data columns (total 2 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   Tweet      451295 non-null  object
 1   Sentiment  451296 non-null  object
dtypes: object(2)
memory usage: 10.3+ MB


In [10]:
df['Sentiment'].value_counts()

Sentiment
Strong_Pos    233674
Neutral        77013
Mild_Pos       64000
Strong_Neg     42555
Mild_Neg       34054
Name: count, dtype: int64

In [11]:
df.isnull().sum()

Tweet        1
Sentiment    0
dtype: int64

In [12]:
df.sample()

,Tweet,Sentiment
294533,"Israelis to join Canada, other countries in 'C...",Strong_Pos


## Before we do any preprocessing , lets just explore our dataset and identify how many impurities we have

In [13]:
import re
import pandas as pd

def check_preprocessing_targets(df, text_col='Tweet', slang_dict=None):
    """
    Checks if original text column contains patterns like emoji, URL, mentions, hashtags, etc.
    Returns a summary count for each pattern.
    """

    url_pattern = re.compile(r"http\S+|www\S+|https\S+")
    mention_pattern = re.compile(r"@\w+")
    hashtag_pattern = re.compile(r"#\w+")
    emoji_pattern = re.compile(
        "[" u"\U0001F600-\U0001F64F"
             u"\U0001F300-\U0001F5FF"
             u"\U0001F680-\U0001F6FF"
             u"\U0001F1E0-\U0001F1FF" "]+", flags=re.UNICODE)
    number_pattern = re.compile(r"\d+")
    date_pattern = re.compile(r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b")
    duration_pattern = re.compile(r"\b\d+\s?(hours?|days?|weeks?|mins?|minutes?|24/7)\b", flags=re.IGNORECASE)
    rating_pattern = re.compile(r"\b\d{1,2}/10\b")
    underscore_pattern = re.compile(r"_")

    def contains_slang(text):
        if slang_dict is None:
            return False
        return any(word in slang_dict for word in str(text).split())

    summary = {
        'URLs': df[text_col].apply(lambda x: bool(url_pattern.search(str(x)))).sum(),
        'Mentions (@)': df[text_col].apply(lambda x: bool(mention_pattern.search(str(x)))).sum(),
        'Hashtags (#)': df[text_col].apply(lambda x: bool(hashtag_pattern.search(str(x)))).sum(),
        'Emojis': df[text_col].apply(lambda x: bool(emoji_pattern.search(str(x)))).sum(),
        'Numbers': df[text_col].apply(lambda x: bool(number_pattern.search(str(x)))).sum(),
        'Dates': df[text_col].apply(lambda x: bool(date_pattern.search(str(x)))).sum(),
        'Durations': df[text_col].apply(lambda x: bool(duration_pattern.search(str(x)))).sum(),
        'Ratings (/10)': df[text_col].apply(lambda x: bool(rating_pattern.search(str(x)))).sum(),
        'Underscores': df[text_col].apply(lambda x: bool(underscore_pattern.search(str(x)))).sum(),
        'Slang': df[text_col].apply(contains_slang).sum() if slang_dict else 'Skipped',
    }

    return pd.DataFrame.from_dict(summary, orient='index', columns=['Count']).sort_values(by='Count', ascending=False)


In [14]:
slang_dict = {
    "u": "you",
    "ur": "your",
    "r": "are",
    "wht":'what',
    "lol": "laughing out loud",
    "lmao": "laughing my ass off",
    "rofl": "rolling on the floor laughing",
    "omg": "oh my god",
    "idk": "i don't know",
    "btw": "by the way",
    "imo": "in my opinion",
    "imho": "in my humble opinion",
    "brb": "be right back",
    "ttyl": "talk to you later",
    "smh": "shaking my head",
    "tbh": "to be honest",
    "dm": "direct message",
    "irl": "in real life",
    "jk": "just kidding",
    "afk": "away from keyboard",
    "np": "no problem",
    "thx": "thanks",
    "ty": "thank you",
    "yw": "you’re welcome",
    "bff": "best friends forever",
    "bday": "birthday",
    "gr8": "great",
    "plz": "please",
    "wtf": "what the fuck",
    "wth": "what the hell",
    "bcoz": "because",
    "coz": "because",
    "cya": "see you",
    "msg": "message",
    "fyi": "for your information",
    "asap": "as soon as possible",
    "atm": "at the moment",
    "bbl": "be back later",
    "cu": "see you",
    "gg": "good game",
    "gl": "good luck",
    "hf": "have fun",
    "hmu": "hit me up",
    "ikr": "i know right"
}

In [15]:
check_preprocessing_targets(df, text_col='Tweet', slang_dict=slang_dict)


,Count
Numbers,280654
URLs,224196
Mentions (@),211497
Hashtags (#),139494
Underscores,32302
Durations,4488
Slang,3164
Dates,973
Ratings (/10),47
Emojis,0


In [16]:
# Step 1: Split first
X = df['Tweet']
y = df['Sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

In [17]:
X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, stratify=y_test, test_size=0.5, random_state=42)

In [18]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)
print(X_val.shape)
print(y_val.shape)

(361036,)
(45130,)
(361036,)
(45130,)
(45130,)
(45130,)


In [19]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)
print(X_val.shape)
print(y_val.shape)

(361036,)
(45130,)
(361036,)
(45130,)
(45130,)
(45130,)


In [20]:
# Step 2: Combine X_train and y_train for sampling
train_df = pd.DataFrame({'Tweet': X_train, 'Sentiment': y_train})

In [21]:
train_df.value_counts('Sentiment')

Sentiment
Strong_Pos    186939
Neutral        61610
Mild_Pos       51200
Strong_Neg     34044
Mild_Neg       27243
Name: count, dtype: int64

In [ ]:
# Sample 5000 examples from training set
train_sample_idx = X_train.sample(n=80000, random_state=1).index
X_train = X_train.loc[train_sample_idx].reset_index(drop=True)
y_train = y_train.loc[train_sample_idx].reset_index(drop=True)

# Sample 1000 examples from validation set
val_sample_idx = X_val.sample(n=4000, random_state=2).index
X_val = X_val.loc[val_sample_idx].reset_index(drop=True)
y_val = y_val.loc[val_sample_idx].reset_index(drop=True)





# Sample 1000 examples from test set
test_sample_idx = X_test.sample(n=7500, random_state=3).index
X_test = X_test.loc[test_sample_idx].reset_index(drop=True)
y_test = y_test.loc[test_sample_idx].reset_index(drop=True)


In [26]:
# Step 2: Combine X_train and y_train for sampling
train_df = pd.DataFrame({'Tweet': X_train, 'Sentiment': y_train})

In [27]:
from sklearn.utils import resample

# Assuming your DataFrame is called `df`
min_class_size = train_df['Sentiment'].value_counts().min()

# Undersample each class to match the smallest class
train_df = (
    train_df.groupby('Sentiment')
      .apply(lambda x: x.sample(n=min_class_size, random_state=42))
      .reset_index(drop=True)
)

C:\Users\harsh\AppData\Local\Temp\ipykernel_36304\3722276867.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min_class_size, random_state=42))


In [28]:
train_df.value_counts('Sentiment')

Sentiment
Mild_Neg      7604
Mild_Pos      7604
Neutral       7604
Strong_Neg    7604
Strong_Pos    7604
Name: count, dtype: int64

In [29]:
# Step 4: Separate again
X_train= train_df['Tweet']
y_train= train_df['Sentiment']

In [30]:
# to comapre results after preprocessing
X_train_backup = X_train.copy()
y_train_backup = y_train.copy()

In [31]:
X_train = pd.DataFrame(X_train)
X_test = pd.DataFrame(X_test)
X_val = pd.DataFrame(X_val)

In [33]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)
print(X_val.shape)
print(y_val.shape)


(38020, 1)
(38020,)
(7500, 1)
(7500,)
(4000, 1)
(4000,)


## Basic PreProcessing

In [28]:
 X_train.head()

,Tweet
0,So many people want to ignore the fact that is...
1,"âPeopleâs Convoyâ of 1,000+ Trucks Gears..."
2,#FreeTheProtestors âFreedom Convoyâ organi...
3,"From Canada, a counter-protestor blocks the ro..."
4,Can we stop what corrupted leaders want to tur...


In [29]:
def clean_tweet(tweet):
    tweet = str(tweet).lower()                               # convert to lowercase
    tweet = re.sub(r"@\S+", '', tweet)                       # remove mentions (@username)
    tweet = re.sub(r"http\S+|www\S+|https\S+", '', tweet)    # remove URLs
    tweet = re.sub(r"#", '', tweet)                          # keep hashtag content
    tweet = re.sub(r"[_\W]+", ' ', tweet)                    # remove underscores & punctuation
    tweet = re.sub(r'\s+', ' ', tweet).strip()               # remove extra spaces
    return tweet


## Numbers Handling

In [30]:
def smart_number_handler(text):

    # Replace emotional ratings like "0/10" or "10/10"
    text = re.sub(r'\b\d{1,2}/10\b', ' <rating> ', text)

    # Replace durations like "3 hours", "2 days", "24/7"
    text = re.sub(r'\b\d+\s?(hours?|days?|weeks?|mins?|minutes?|24/7)\b', ' <duration> ', text, flags=re.IGNORECASE)

    # Replace dates like "12/03/2022" or "1-1-21"
    text = re.sub(r'\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b', ' <date> ', text)

    # Replace percentages like "90%"
    text = re.sub(r'\b\d+%|\d+\spercent\b', ' <percent> ', text, flags=re.IGNORECASE)

    # Replace currency values like "$200", "€30", "£50.00"
    text = re.sub(r'[$€£]\s?\d+(\.\d+)?', ' <money> ', text)

    # Remove numbers expect for years 1900 - 2100
    text = re.sub(r'\b(?!19\d{2}\b|20\d{2}\b|2100\b)\d+\b', '', text)

    return text


## Emoji -> Text emotions (like happy_smiley_face)

In [31]:
pip install emoji

In [32]:
import emoji

def convert_emojis_to_text(text):
    return emoji.demojize(text, language='en')

def simplify_emoji_text(text):
    text = convert_emojis_to_text(text)
    text = text.replace(":", "")  # remove colons
    text = text.replace("_", " ")  # optional: make it tokenizer friendly
    return text


## Slang and Abbrevation *Dictionary*

## Slang replaceing

In [33]:
def replace_slangs(text):
    words = text.split()
    return ' '.join([slang_dict[word] if word in slang_dict else word for word in words])

## Spell Check

In [34]:
from spellchecker import SpellChecker
spell = SpellChecker()  
def correct_text_spelling(text):
    try:
        words = text.split()
        corrected_words = [spell.correction(word) if word.isalpha() else word for word in words]
        return ' '.join(corrected_words)
    except Exception:
        return text

## Negation Handeling

In [35]:
import spacy


In [36]:
# Load the SpaCy English model
nlp = spacy.load("en_core_web_sm")

def handle_negation(text):
    """
    Add 'NOT_' prefix to words that are negated.
    """
    
    doc = nlp(text)
    negated = set()

    # Identify the heads of negation dependencies
    for token in doc:
        if token.dep_ == "neg":
            negated.add(token.head.i)

    # Apply NOT_ prefix to negated tokens
    result = []
    for i, token in enumerate(doc):
        if i in negated:
            result.append("NOT_" + token.text)
        else:
            result.append(token.text)

    return " ".join(result)

In [37]:
tweet1 = "I don't like the food."
tweet2 = "They never support the idea."
tweet3 = "This is not bad at all."

print(handle_negation(tweet1))  # I do not NOT_like the food .
print(handle_negation(tweet2))  # They never NOT_support the idea .
print(handle_negation(tweet3))  # This is not NOT_bad at all .


I do n't NOT_like the food .
They never NOT_support the idea .
This NOT_is not bad at all .


## Main Pipeline Function

In [38]:
def preprocessing_pipeline(text):
    if not isinstance(text, str):
        return ""

    text = text.lower().strip()  # Optional: Normalize casing & whitespace early

    text = smart_number_handler(text)              # Convert smart numbers to textual forms
    text = simplify_emoji_text(text)               # Convert emojis to meaningful words (e.g., 😊 → "smile")
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)     # Remove non-ASCII leftovers (e.g., fancy quotes, symbols)
    text = clean_tweet(text)                       # Remove mentions, hashtags, punctuation, excess spaces
    text = handle_negation(text)                   # Tag negations BEFORE correction to preserve context
    text = correct_text_spelling(text)             # Fix misspellings (after negations so “nott” becomes “not”)
    text = replace_slangs(text)                    # Replace informal words (after spelling fix for max match)
    text = smart_number_handler(text)              # Run again if slang/spell fix introduces numbers

    return text

In [ ]:


tweets = X_train['Tweet'].astype(str).tolist()

# Use threads instead of processes
processed_tweets = Parallel(n_jobs=-1, prefer="threads")(
    delayed(preprocessing_pipeline)(tweet) for tweet in tqdm(tweets)
)

X_train['Cleaned_Tweet'] = processed_tweets

  0%|          | 89/18980 [00:43<2:26:23,  2.15it/s]

## Lemmatization and Stop words

In [ ]:
#  Download once, outside of the function
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\harsh\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\harsh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\harsh\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\harsh\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [ ]:
# Helper for POS
def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN  # default to noun

In [ ]:
# Now safe for multiprocessing
def lemmatize_and_filter(text):
    if not isinstance(text, str):
        return ""

    tokens = word_tokenize(text)
    tagged_tokens = pos_tag(tokens)

    cleaned_tokens = []
    for token, tag in tagged_tokens:
        token_lower = token.lower()
        if token_lower not in stop_words or token.startswith("NOT_"):
            lemma = lemmatizer.lemmatize(token_lower, get_wordnet_pos(tag))
            cleaned_tokens.append(lemma)

    return " ".join(cleaned_tokens)


In [ ]:
sample_text = "I do NOT_like eating apples because they are NOT_good and sometimes make me feel tired."

result = lemmatize_and_filter(sample_text)
print("Original:", sample_text)
print("Lemmatized & Stopword-filtered:", result)


Original: I do NOT_like eating apples because they are NOT_good and sometimes make me feel tired.
Lemmatized & Stopword-filtered: not_like eat apple not_good sometimes make feel tired .


In [ ]:
# Convert column to list
texts = X_train['Cleaned_Tweet'].astype(str).tolist()

# Use threads for lightweight operations
lemmatized = Parallel(n_jobs=-1, prefer="threads")(
    delayed(lemmatize_and_filter)(text) for text in tqdm(texts)
)

# Store results
X_train['Final_Cleaned_Tweet'] = lemmatized

100%|██████████| 235/235 [00:00<00:00, 1161.17it/s]


In [ ]:
X_train.head().to_csv('sample.csv',index=False)

In [ ]:
# ----------- For Test Set -------------
# Preprocessing
test_tweets = X_test['Tweet'].astype(str).tolist()
processed_test = Parallel(n_jobs=-1, prefer="threads")(
    delayed(preprocessing_pipeline)(tweet) for tweet in tqdm(test_tweets)
)
X_test['Cleaned_Tweet'] = processed_test

# Lemmatization
lemmatized_test = Parallel(n_jobs=-1, prefer="threads")(
    delayed(lemmatize_and_filter)(text) for text in tqdm(X_test['Cleaned_Tweet'].astype(str))
)
X_test['Final_Cleaned_Tweet'] = lemmatized_test

100%|██████████| 100/100 [00:00<00:00, 1436.43it/s]


In [ ]:
# ----------- For Validation Set -------------
# Preprocessing
val_tweets = X_val['Tweet'].astype(str).tolist()
processed_val = Parallel(n_jobs=-1, prefer="threads")(
    delayed(preprocessing_pipeline)(tweet) for tweet in tqdm(val_tweets)
)
X_val['Cleaned_Tweet'] = processed_val

# Lemmatization
lemmatized_val = Parallel(n_jobs=-1, prefer="threads")(
    delayed(lemmatize_and_filter)(text) for text in tqdm(X_val['Cleaned_Tweet'].astype(str))
)
X_val['Final_Cleaned_Tweet'] = lemmatized_val

100%|██████████| 100/100 [00:00<00:00, 1421.27it/s]


In [ ]:
X_val.head()

,Tweet,Cleaned_Tweet,Final_Cleaned_Tweet
0,I love the idea of US truckers pulling a #Free...,i love the idea of us truckers pulling a freed...,love idea u trucker pull freedomconvoy think r...
1,å æ¿å¤§æ æ¥é¨é¨ä¸æ¥åç¹é²å¤æå ³â...,registered registered sis memo rejects trudeau...,register registered si memo reject trudeau fre...
2,Freedom Convoy 2022\nhttps://t.co/sdUPlfSUBi h...,freedom convoy 2022,freedom convoy 2022
3,@dbeggs13 Could have been wrapped up a year ag...,could have been wrapped up a year ago if or on...,could wrap year ago one minister meet proteste...
4,CANADA &amp;USA: Nationwide Convoy-Freedom-Pro...,canada amp usa nationwide convoy freedom prote...,canada amp usa nationwide convoy freedom prote...


In [ ]:
X_train.to_csv('X_train.csv',index=False)
X_test.to_csv('X_test.csv',index=False)
X_val.to_csv('X_val.csv',index=False)
y_train.to_csv('y_train.csv',index=False)
y_test.to_csv('y_test.csv',index=False)
y_val.to_csv('y_val.csv',index=False)